# Environment and model setup

This notebook inspects configuration without starting a GPU process. Follow the adjacent README for environment creation and hash-checked model restoration.

In [ ]:
from pathlib import Path
import sys, json
YOUR_PATHWAY = "YOUR PATHWAY"
if YOUR_PATHWAY == "YOUR PATHWAY":
    ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p/"Models/adapter_manifest.json").is_file()), None)
    if ROOT is None:
        raise ValueError("Set YOUR_PATHWAY to the repository clone directory.")
else:
    ROOT = Path(YOUR_PATHWAY).expanduser().resolve()
sys.path[:0] = [str(ROOT/"Experiment Codes/src"), str(ROOT/"Parallel Agent Setup/src")]
RUN_INFERENCE = False
RUN_TRAINING = False


In [ ]:
for family in ["Mistral", "Qwen"]:
    manifest = json.loads((ROOT/"Models"/family/"base_manifest.json").read_text())
    print(family, manifest["model"], manifest["revision"])
    print("Base weight bytes:", sum(x["bytes"] for x in manifest["files"] if x["path"].endswith(".safetensors")))


Restore adapters using the private release and `tools/fetch_models.py`. Start one family at a time; a fresh process does not inherit another manager's conversation.

In [ ]:
import importlib.util
spec = importlib.util.spec_from_file_location("server_setup", ROOT/"Parallel Agent Setup/serve.py")
server = importlib.util.module_from_spec(spec); spec.loader.exec_module(server)
FAMILY = "Mistral"
server.start(FAMILY, execute=RUN_INFERENCE)
